Program ma przyjmować (input) zdanie z błędnie zapisanymi słowami (można zahardkodować, a nie wpisywać przy każdym uruchomieniu).

Np. "Kotty to fajne zfieszęta". Proszę też wymyślić własne przykłady.

TODO:

- preprocessing
- sprawdzić każde ze słów z SJP (sjp.pl) czy jest poprawne,
- dla niepoprawnych słów sprawdzić odległość edycyjną do wszystkich słów z SJP i wybrać 3-5 najlepszych (najmniejsza odległość nie zawsze jest najlepsza, czasem lepiej wybrać 10-20 i odsiać później),
- na podstawie tekstów z korpusu sprawdzić które z proponowanych poprawek pojawiają się w zdaniach z wyrazami poprawnymi (np. zwierzęta i koty).
- wyświetlić zdanie z poprawkami - "Koty to fajne zwierzęta".

In [2]:
import marshal
import re
import os
from collections import Counter, defaultdict

def levenshtein(word1, word2):
    n, m = len(word1), len(word2)
    if n == 0: return m
    if m == 0: return n

    res = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): res[i][0] = i
    for j in range(m + 1): res[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if word1[i-1] == word2[j-1] else 1
            res[i][j] = min(res[i-1][j] + 1,
                            res[i][j-1] + 1,
                            res[i-1][j-1] + cost)
    return res[n][m]

def words_from_line(line):
    return re.findall(r"(\w+)([^\w]*|$)", line.lower())

def sjp_save():
    sjp = set()
    with open("odm.txt", "r", encoding="utf-8") as file:
        for f in file:
            words = f.strip().lower().split(",")
            for w in words:
                sjp.add(w.strip())  
    with open("sjp", "wb") as dfile:
        marshal.dump(sjp, dfile)
        
def sjp_load():
    with open("sjp", "rb") as file:
        return marshal.load(file)

def sjp_check(line_words, sjp, max_dist=3, max_proposals=5):
    incorrect_words = {}
    for word_tuple in line_words:
        word = word_tuple[0]
        
        if not word or word in sjp:
            continue

        distances = []
        len_filter = 3  
        for w in sjp:
            
            if abs(len(word) - len(w)) > len_filter:
                continue
        
            dist = levenshtein(word, w)
            if dist <= max_dist:
                distances.append((dist, w))
        
        distances.sort(key=lambda x: (x[0], x[1]))
        incorrect_words[word] = [d[1] for d in distances[:max_proposals]]
    return incorrect_words

def load_corpus_cooccurrences(folder="TXT"):
    cooccurrences = defaultdict(Counter)
    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            filepath = os.path.join(folder, filename)
            with open(filepath, "r", encoding="utf-8") as file:
                text = file.read().lower()
                sentences = re.split(r'[.!?]', text) 
                for sentence in sentences:
                    words = re.findall(r"\w+", sentence)
                    for i in range(len(words) - 1):
                        w1 = words[i]
                        w2 = words[i+1]
                        cooccurrences[w1][w2] += 1
                        cooccurrences[w2][w1] += 1
    return cooccurrences

def contextual_correction(original_sentence, sjp, cooccurrences):
    line_words = words_from_line(original_sentence)
    incorrect_proposals = sjp_check(line_words, sjp)
    context_words = {word_tuple[0] for word_tuple in line_words 
                     if word_tuple[0] in sjp and word_tuple[0] not in incorrect_proposals}
    best_corrections = {}
    
    for incorrect_word, proposals in incorrect_proposals.items():
        scored_proposals = []
        for proposal in proposals:
            context_score = 0
            for context_word in context_words:
                score_a_b = cooccurrences[proposal].get(context_word, 0)
                score_b_a = cooccurrences[context_word].get(proposal, 0)
                context_score += score_a_b + score_b_a

            dist = levenshtein(incorrect_word, proposal)
            scored_proposals.append((context_score, -dist, proposal)) 
        scored_proposals.sort(key=lambda x: (x[0], x[1]), reverse=True)

        if scored_proposals:
            best_proposal = scored_proposals[0][2]
            best_corrections[incorrect_word] = best_proposal
        else:
            best_corrections[incorrect_word] = proposals[0] if proposals else incorrect_word
            
        print(f"Propozcje zmain: {[ (s[2], s[0]) for s in scored_proposals[:5] ]}")
        print(f"Dla '{incorrect_word}' najlepsza poprawka to '{best_corrections[incorrect_word]}'")
    
    corrected_sentence_parts = []
    for word, punctuation in line_words:
        corrected_word = best_corrections.get(word, word)
        corrected_sentence_parts.append(corrected_word + punctuation)
        
    corrected_sentence = "".join(corrected_sentence_parts).capitalize()
    
    return corrected_sentence, best_corrections



SJP_WORDS = sjp_load()
CORPUS_COOCCURRENCES = load_corpus_cooccurrences("TXT")

test_sentences = [
    "Kotty to fajne zfieszęta.",
    "Dzisiai póidę zjeść do miasta.",
]

for idx, sentence in enumerate(test_sentences):
    print(f"\nZdanie {idx+1}: '{sentence}'")
    
    corrected_sentence, corrections_map = contextual_correction(sentence, SJP_WORDS, CORPUS_COOCCURRENCES)
    
    print(f"Poprawione zdanie: {corrected_sentence}")
    print("\n" + "-"*50)


Zdanie 1: 'Kotty to fajne zfieszęta.'
Propozcje zmain: [('zwierzęta', 0), ('fiesta', 0), ('nieszyta', 0), ('zbeszta', 0), ('zbieszona', 0)]
Dla 'zfieszęta' najlepsza poprawka to 'zwierzęta'
Poprawione zdanie: Kotty to fajne zwierzęta.

--------------------------------------------------

Zdanie 2: 'Dzisiai póidę zjeść do miasta.'
Propozcje zmain: [('dzisiaj', 40), ('dzikimi', 0), ('dzisna', 0), ('dziwili', 0), ('zdzisia', 0)]
Dla 'dzisiai' najlepsza poprawka to 'dzisiaj'
Propozcje zmain: [('pójdę', 0), ('aidę', 0), ('alidę', 0), ('amidę', 0), ('aoidę', 0)]
Dla 'póidę' najlepsza poprawka to 'pójdę'
Poprawione zdanie: Dzisiaj pójdę zjeść do miasta.

--------------------------------------------------
